In [0]:
# VoeBem Analytics AI
# Camada Bronze - VRA (Voo Regular Ativo)
# Fonte: ANAC - Dados Abertos

CATALOG = "voebem"
SCHEMA_BRONZE = "bronze"

VOLUME_VRA = "/Volumes/voebem/bronze/arquivos"

print(f"Fonte VRA: {VOLUME_VRA}")

In [0]:
# Lista os arquivos disponíveis no Volume da ANAC

arquivos = dbutils.fs.ls(VOLUME_VRA)

arquivos_csv = [
    arquivo
    for arquivo in arquivos
    if arquivo.name.lower().endswith(".csv")
]

print(f"Total de arquivos CSV encontrados: {len(arquivos_csv)}")

for arquivo in sorted(arquivos_csv, key=lambda x: x.name):
    print(arquivo.name)

In [0]:
# Leitura inicial dos arquivos VRA da ANAC
# Nesta etapa ainda não fazemos transformações.

df_vra_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .option("encoding", "UTF-8")
    .csv(f"{VOLUME_VRA}/*.csv")
)

print(f"Quantidade de registros: {df_vra_raw.count():,}")
print(f"Quantidade de colunas: {len(df_vra_raw.columns)}")

df_vra_raw.printSchema()

In [0]:
# Inspeciona as primeiras linhas de um arquivo VRA
# para identificar a estrutura real do CSV da ANAC.

arquivo_teste = f"{VOLUME_VRA}/VRA_20258.csv"

linhas = (
    spark.read
    .text(arquivo_teste)
    .limit(5)
    .collect()
)

for i, linha in enumerate(linhas, start=1):
    print(f"Linha {i}: {linha['value']}")

In [0]:
# Teste de leitura correta de um arquivo VRA
# A primeira linha contém metadados ("Atualizado em...")
# e deve ser ignorada.

df_teste = (
    spark.read
    .option("header", "true")
    .option("sep", ";")
    .option("quote", '"')
    .option("encoding", "UTF-8")
    .option("inferSchema", "false")
    .option("skipRows", 1)
    .csv(arquivo_teste)
)

print(f"Registros: {df_teste.count():,}")
print(f"Colunas: {len(df_teste.columns)}")

df_teste.printSchema()

In [0]:
# Ingestão dos 12 arquivos VRA
# Agosto/2025 a Julho/2026

df_vra_bronze = (
    spark.read
    .option("header", "true")
    .option("sep", ";")
    .option("quote", '"')
    .option("encoding", "UTF-8")
    .option("inferSchema", "false")
    .option("skipRows", 1)
    .csv(f"{VOLUME_VRA}/*.csv")
)

print(f"Registros totais: {df_vra_bronze.count():,}")
print(f"Colunas: {len(df_vra_bronze.columns)}")

df_vra_bronze.printSchema()

In [0]:
from pyspark.sql.functions import col, current_timestamp

# Releitura dos arquivos já incluindo o metadado de origem
df_vra_bronze = (
    spark.read
    .option("header", "true")
    .option("sep", ";")
    .option("quote", '"')
    .option("encoding", "UTF-8")
    .option("inferSchema", "false")
    .option("skipRows", 1)
    .csv(f"{VOLUME_VRA}/*.csv")
    .select(
        "*",
        col("_metadata.file_path").alias("_arquivo_origem")
    )
    .withColumn("_data_ingestao", current_timestamp())
)

print(f"Registros: {df_vra_bronze.count():,}")
print(f"Colunas após metadados: {len(df_vra_bronze.columns)}")

display(
    df_vra_bronze.select(
        "_arquivo_origem",
        "_data_ingestao"
    ).limit(10)
)

In [0]:
# Normalização dos nomes das colunas para persistência Delta

nomes_colunas = {
    "ICAO Empresa Aérea": "icao_empresa_aerea",
    "Número Voo": "numero_voo",
    "Código Autorização (DI)": "codigo_autorizacao_di",
    "Código Tipo Linha": "codigo_tipo_linha",
    "ICAO Aeródromo Origem": "icao_aerodromo_origem",
    "ICAO Aeródromo Destino": "icao_aerodromo_destino",
    "Partida Prevista": "partida_prevista",
    "Partida Real": "partida_real",
    "Chegada Prevista": "chegada_prevista",
    "Chegada Real": "chegada_real",
    "Situação Voo": "situacao_voo",
    "Código Justificativa": "codigo_justificativa"
}

for antigo, novo in nomes_colunas.items():
    df_vra_bronze = df_vra_bronze.withColumnRenamed(antigo, novo)

TABELA_BRONZE_VRA = f"{CATALOG}.{SCHEMA_BRONZE}.vra"

(
    df_vra_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABELA_BRONZE_VRA)
)

print(f"Tabela criada: {TABELA_BRONZE_VRA}")
print(f"Registros gravados: {spark.table(TABELA_BRONZE_VRA).count():,}")